# Look at mapping breadth and depth of 100 x metagenomes to singleclust genes

In [2]:
import polars as pl
import glob
import os
import screed
import csv
import screed

## Read in individual metag x species singleclust depth reports

These files contain depth-of-mapping information for all our genes.

In [3]:
DIR='../outputs.cds/singleclust/bam.bak'
template = '../outputs.cds/singleclust/bam/{metag}.x.{species}.depth.txt'

def read_depth_txt(metag, species, *, exclude_ends=75):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo')).select(['gene', 'pos', 'cov'])

    sum_df = df.group_by('gene').all().with_columns(
        # select slice [75:-75]
        (pl.col("pos").list.slice(exclude_ends, -exclude_ends).list.len()).alias("len"),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends)),
        (pl.col("cov").list.slice(exclude_ends, -exclude_ends).list.filter(pl.element() > 0)).list.len().alias("hits"),
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species")),
        # summarize: average depth across contig,
        (pl.col("cov").list.sum() / pl.col("len")).alias("depth_all"),
        # average depth across covered bases,
        (pl.col("cov").list.sum() / pl.col("hits")).alias("depth_cov"),
        # fraction of bases covered
        (pl.col("hits") / pl.col("cov").list.len()).alias("breadth"),
    ).select(["metag", "species", "gene", "len", "hits", "breadth", "depth_all", "depth_cov"])
    return sum_df

read_depth_txt('ERR1135199', 's__Cryptobacteroides sp900546925')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""GDFPIIAG_01475""",276,224,0.811594,1.144928,1.410714
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""NMJBBHDE_00454""",771,552,0.715953,1.250324,1.746377
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""MFHJPMDA_01126""",2031,1014,0.499261,4.707041,9.428008
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""FNIGJHLI_01889""",1182,446,0.377327,0.683587,1.811659
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""CPBNAKKL_01716""",261,205,0.785441,1.429119,1.819512
…,…,…,…,…,…,…,…
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""GIIIOOFK_00161""",930,0,0.0,0.0,NaN
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""GGBBOKIP_00385""",450,253,0.562222,1.12,1.992095
"""ERR1135199""","""s__Cryptobacteroides sp9005469…","""PGBAIDML_01792""",1365,0,0.0,0.0,NaN


In [27]:
# Read them all in!
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth_txt(metag, species))

depth_df = pl.concat(dflist)

print(f"read {len(filenames)} depth files.")

0 of 1400
100 of 1400
200 of 1400
300 of 1400
400 of 1400
500 of 1400
600 of 1400
700 of 1400
800 of 1400
900 of 1400
1000 of 1400
1100 of 1400
1200 of 1400
1300 of 1400
read 1400 depth files.


In [28]:
depth_df.filter(pl.col('depth_all').is_not_nan()).sort(by='depth_all', descending=True).filter(pl.col('depth_all') > 0.0)

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64
"""SRR14369134""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,17800.038462,17800.038462
"""SRR11489750""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,12280.894587,12280.894587
"""SRR12795790""","""s__Sodaliphilus sp004557565""","""JHPFMEME_00905""",702,702,1.0,8948.836182,8948.836182
"""SRR10209683""","""s__Mogibacterium_A kristiansen…","""AAOFCIJB_00277""",1119,1119,1.0,8170.679178,8170.679178
"""SRR17241663""","""s__Mogibacterium_A kristiansen…","""JMKEGFFJ_01326""",1899,1899,1.0,7469.317536,7469.317536
…,…,…,…,…,…,…,…
"""ERR3211876""","""s__Gemmiger qucibialis""","""PMKFNJDI_00477""",3072,4,0.001302,0.001302,1.0
"""SRR8655118""","""s__Cryptobacteroides sp9005469…","""CMHHCIHA_00893""",2016,2,0.000992,0.000992,1.0
"""SRR11124687""","""s__Prevotella sp002251295""","""OPHKNFIJ_00863""",2094,2,0.000955,0.000955,1.0


## Summarize our mapping breadth results across all the metagenomes

In [29]:
# require 10% of each gene to be covered by at least one read
BREADTH_CUTOFF = 0.1

In [30]:
# aggregate across all metagenomes;
# calculate fraction of metagenomes for which gene mapping exceeds our breadth cutoff
agg_df = depth_df.group_by(['species', 'gene']).agg(
    # get fraction of columns where breadth is greater than cutoff as 'f'
    ((pl.col("breadth") >= BREADTH_CUTOFF).sum() / pl.col("breadth").len()).alias("f"),
#    (pl.col("depth").filter(pl.col("depth").is_not_nan()).mean()),
)
agg_df

species,gene,f
str,str,f64
"""s__Sodaliphilus sp004557565""","""MLIHAKPI_01533""",0.91
"""s__Sodaliphilus sp004557565""","""ANJDDJEL_00841""",0.8
"""s__Cryptobacteroides sp9005469…","""EHLNOCPD_00320""",0.7
"""s__Cryptobacteroides sp9005469…","""FNIGJHLI_00117""",0.83
"""s__Bariatricus sp004560705""","""LEHFNBLN_02301""",0.93
…,…,…
"""s__Sodaliphilus sp004557565""","""AONHLPKD_00542""",0.88
"""s__Sodaliphilus sp004557565""","""DDMOEMBK_01105""",0.69
"""s__Prevotella sp002251295""","""IOPEOCKD_01198""",0.89


In [31]:
# print information out by species
for species in sorted(agg_df['species'].unique()):
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    foo_df = foo_df.sort(by='f', descending=True).filter(pl.col('f') > 0.8)
    print(foo_df)

    top50_names = set(foo_df.head(50)['gene'].to_list())

    outfile = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.fa'
    print(outfile)
    outfp = open(outfile, 'wt')
    for record in screed.open(f'../outputs.cds/singleclust/{species}.cds3.min50.dedup.fa'):
        name = record.name.split(' ')[0]
        if name in top50_names:
            top50_names.remove(name)
            outfp.write(f'>{record.name}\n{record.sequence}\n')
    assert not top50_names
    outfp.close()

    outfile2 = f'../outputs.cds/singleclust/{species}.cds.min50.dedup.top50.csv'
    print(outfile2)
    outfp = open(outfile2, 'w', newline='')
    w = csv.writer(outfp)

    for name in foo_df.head(50)['gene'].to_list():
        w.writerow(['0', species, name, "(not reviewed)"])
    outfp.close()
            
    

s__Bariatricus sp004560705
shape: (14, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ NDCIOGGG_00214 ┆ 0.9  │
│ s__Bariatricus sp004560705 ┆ FMMMFAAO_00746 ┆ 0.89 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ IEPLHDOE_02248 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ NOACAEKI_00860 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ NGBMPMHL_00028 ┆ 0.81 │
└────────────────────────────┴────────────────┴──────┘
../outputs.cds/singlecl

## Explore specific gene/species/metag combinations

In [32]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth')

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64


In [33]:
depth_df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth').filter(pl.col("metag") == "ERR1135199")

metag,species,gene,len,hits,breadth,depth_all,depth_cov
str,str,str,u32,u32,f64,f64,f64


## Try something else

In [34]:
species_genes_df = (pl.read_csv('../outputs.cds/singleclust/species-genes.csv')
    .filter(pl.col("good") == 1)
    .with_columns(pl.col('gene_name').alias('gene'))
    .select(["anchor", "gene", "species", "description"])
)

species_genes_df

anchor,gene,species,description
i64,str,str,str
0,"""CNENGHLA_01260""","""s__Phascolarctobacterium_A suc…","""BLAST match to hydrogenase lar…"
1,"""EHOAPHDI_01174""","""s__Phascolarctobacterium_A suc…","""BLAST match to protein phospha…"
0,"""CNENGHLA_00658""","""s__Phascolarctobacterium_A suc…","""BLAST match to 4-hydroxy-3-met…"
0,"""IFIBFMPA_00800""","""s__Phascolarctobacterium_A suc…","""BLAST match to 2-isopropylmal…"
0,"""BBOFCOCJ_01349""","""s__Lactobacillus amylovorus""","""BLAST match to peptidase T [La…"
…,…,…,…
0,"""COCKKAPE_02026""","""s__Prevotella sp002251295""","""polysaccharide biosynthesis C-…"
0,"""KINAFDOL_01550""","""s__Prevotella sp002251295""","""transcription-repair coupling …"
0,"""KPACIEHJ_00390""","""s__Prevotella sp002251295""","""DUF4302 protein"""


In [35]:
merge_df = depth_df.join(species_genes_df, on=["species", "gene"], how='inner')
merge_df

metag,species,gene,len,hits,breadth,depth_all,depth_cov,anchor,description
str,str,str,u32,u32,f64,f64,f64,i64,str
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""IEPMJGAI_00771""",420,115,0.27381,0.27381,1.0,0,"""BLAST match to isolate genome"""
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""LJHBCGLM_01294""",393,0,0.0,0.0,NaN,0,"""BLAST match to isolate genome"""
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""DKAPMIEG_00121""",564,127,0.225177,0.381206,1.692913,1,"""BLAST match to isolate genome"""
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""BHHJJAAP_00933""",531,0,0.0,0.0,NaN,0,"""BLAST match to isolate genome"""
"""ERR1135199""","""s__Mogibacterium_A kristiansen…","""EBLLCBKE_00022""",459,91,0.198257,0.198257,1.0,0,"""BLAST match to isolate genome"""
…,…,…,…,…,…,…,…,…,…
"""ERR8314743""","""s__Lactobacillus amylovorus""","""OMBJBIHD_01711""",1134,1121,0.988536,6.604938,6.681534,1,"""BLAST match to LD-transpeptida…"
"""ERR8314743""","""s__Lactobacillus amylovorus""","""OBGMJDLM_01221""",2574,2574,1.0,12.188034,12.188034,0,"""BLAST match to calcium-translo…"
"""SRR12795793""","""s__Bariatricus sp004560705""","""CHOLCMOC_00863""",1386,1194,0.861472,1.837662,2.133166,0,"""BLAST match to BREX system Lon…"


In [24]:
#names = set()
#for filename in glob.glob('../outputs.cds/cds3-genes/*.fa'):
#    names.update([ record.name.split(' ')[0] for record in screed.open(filename) ])
#names

In [25]:
#depth_df.filter(pl.col("gene").is_in(names)).sort(by='depth_all')

In [36]:
merge_df.write_csv('../outputs.cds/cds3-genes/mapping-coverage.csv')